# AuraGateway P0-P2 source materializer V2

CPU-only publisher. Use Accelerator None, Internet Off, no secrets, and Save & Run All exactly once.


In [ ]:
from __future__ import annotations

import base64
import hashlib
import io
import json
import stat
import zipfile
from pathlib import Path, PurePosixPath

NOTEBOOK_NAME = "ag-cu129-p0-p2-source-materializer-v2"
OUTPUT_DATASET_NAME = "ag-cu129-p0-p2-source-v2"
OUTPUT_DIRECTORY_NAME = "ag_cu129_p0_p2_source_materializer_v2_output"
SOURCE_BUNDLE_NAME = "ag-cu129-p0-p2-source-bundle-v2.zip"
EXPECTED_SOURCE_BUNDLE_SHA256 = "bd097235f858500adcce86ddc99d15c9e01c2848fc3fece9a4ea587a0e66b88b"
EXPECTED_BUNDLE_MANIFEST_SHA256 = "d99fb24c17a3e2100cda865d9833fbd92876788442ae5c88a305c28798f68cdb"
EXPECTED_SOURCE_INVENTORY_SHA256 = (
    "462dbbaf58d6e0bd1568b5ff712955bc452cd56bfeaea0f5d75a3ba0fb657ff7"
)
EXPECTED_SOURCE_REPOSITORY_COMMIT = "c3af648ab804629ebac41f2e96b1fcfa81baccd1"
BUNDLE_MANIFEST_NAME = "bundle_manifest.json"
SOURCE_INVENTORY_NAME = "source_inventory.json"
SHA256_MANIFEST_NAME = "sha256_manifest.json"
MATERIALIZATION_RECEIPT_NAME = "materialization_receipt.json"
WORK_ROOT = Path("/kaggle/working").resolve()
OUTPUT_ROOT = WORK_ROOT / OUTPUT_DIRECTORY_NAME
STAGING_ROOT = WORK_ROOT / f".{OUTPUT_DIRECTORY_NAME}.staging"
SOURCE_BUNDLE_B64 = (
    "UEsDBBQAAAAIAAAAIQBdBJnigQMAALAHAABCAAAAYXVyYWdhdGV3YXlfY3UxMjlfcDBfcDJf"
    "cGxhdGZvcm1fZGlhZ25vc3RpY19pbXBsZW1lbnRhdGlvbl92MS5qc29uhVXbjts2EP0VQs+x"
    "V747+xa0fgi66ywKFyhQFARFjiRmeVFIyhtvkH/PUJQs2202T7aoM2cOz1z0LYOjFGA40FfZ"
    "UMM0ZPcZqya8nc3fT5p80swnjWKhtE5PBuzkOJsiPHuXwVfgbZDWUG5bEzxtHHgwIbsvmfLw"
    "LpO6UaDxhHUoj7+txxQfH58edo+7/WH3O91/OtDd37vf/sIH5DTwNdCKhajEYUp4ocwIqvHE"
    "SabkK9Amp82cDrqokKwy1gfJY3gUo5jUmOaf7FADeconT3NibIDC2mdSMx8fSAFgSLoAiClG"
    "RixvnUO15A9WVQqI1KyC64hoQZClTDEP0jyDmxyllwXilSx4K9h1ROPsEUxEfyBaGuRU5OBk"
    "sIZgrAF1DedWN1KNkhppDAji0F+pb8UYdFQhmBQnEmrpiYPGeuR2J8JrZioYaLQVt5mUZSLl"
    "2VtyfHh4JC/WoaQO1SGQ3YUzJFE4+NKCDyOoARfLcIYV2CO1Zu6ZBMc+A+/EnNFXjv/58fBp"
    "Tz8cDvtrZUygMBIs4QzrKTk61ktjrjrG0J05SmdNbC3ypcW2KBEVe+ymvniDQklfp4RPzoqW"
    "dzAHTEgD3r8R8G9sp9Q2lINSqcuz+/nF+S9mZuxNnJrsIq5hoca44dnfsdax2PYv7EQ7qp+3"
    "OcUBlM3JFJeEvmbz1RopOYOyLGfL9XaV58DFZgWCi3y+XRQLsc3XbLUpNmt4z3K2EMCLxWxb"
    "8mW+XeC7YlOukNS2oWlDd+f/zrQDbp2gUsRrj6J/ef/rXZDs6JtpcEOwwO7giHnuzl10B2Op"
    "J1elRoo726T184ZZfY7pZ29Nn1I6EDRdslsT57A03eFE4xy5c0wcaiqcPIKjqhv5G0A/1zR0"
    "c03TXN9gRqn/I9K3Gi97GrBFa4QCXHpGlhfa6xZPBlotYoP2e6G/DVK6NHHXxaFHpXTfVi81"
    "gKpt64F2NfCshID4bxgQauvka1rW0vsWxLns54LQcawHb7FJhiWQ3efoF/objcRIinlGEt76"
    "YDXaFyt9/QrXPrqGHvoGTKKpmpaOn5iLFH3Ic7el34R0O4umTXdz+BPtg6Fp4VBvW4efx7RM"
    "R4rhbVqQ/fF3NJPXoBnFRvEoCIswm+bTPLqceDSThqIBFVJarSWOVVYuy3wLgi2LpVhulpvV"
    "DP+tl+sSVov1eim2bLPlbC2y7z8AUEsDBBQAAAAIAAAAIQDi/7KxxiUAANOzAAA0AAAAYXVy"
    "YWdhdGV3YXlfY3UxMjlfcDBfcDJfcGxhdGZvcm1fZGlhZ25vc3RpY192MS5pcHluYu197XLj"
    "uLHo/30Kxvmx0kbSSLIsyz5x6mpszYxqbcuR5Ul2bRdDkZDNDEUqJOWx4uOq8xDnCe+T3G58"
    "kCAJUJRnUnty60yyMzaJbgANoL/Q3Xz5wdiziedFe8fG7Q+G8QL/sSdmvFkReLq3tMIvTvDV"
    "32vQd0sSW44VW/Dq5ZU9ioJ1aBOOAX7/vTFch9ZHKyZfrY1xenM2NDrd1pFx1W5edY0rz4oX"
    "Qbg0zlzrwQ+i2LXvOHJjL/3pInCI11yEhBiTVewGvnFqrASok4C2jLPA8IPYsBzHWCJMw/ga"
    "hF9I2DB8EuOPDcMOiUP82LXgZRAac+LbjzgtI1iR0ELkUQt6xo7v4S86LQUlbEDPqUCeib1G"
    "QNMO1n4Mb/2152kpFKzj1TqmRL5X02wRBkvDNBfreB0S0zTc5SoIYVY+zI2NUEUk3srGEUYt"
    "GJBXePfgBfPCw0crevTc4nP2D7xpiUmUNFF29/co8AsPg6jwSCxl4UX0qMQbreerMLBJVEQV"
    "bYrP/umuFq5H0ueUvjAhErtLIqh7MzttJA9zbVdWjCQSTa/gV9UKXJ9+Gl0Mzc+j6fV4cmmc"
    "GHd7nVa71b6T2pyNhx8vJ9ez8ak5PqNNLDgfD+x8NO11p3vUXLWbq25TUKWZbvDmU0fGdT25"
    "mZ6OzIvh+NK8GE0/jszTycXFeEbRLnqL9oA4Vm/ec3qHvcODDvzU7/UX5GC/3+85A+twYFt9"
    "R0Y4vbmcjS9G5uRmdnUzM8/G09HpbDL9JT9O88nzliYdrPn1kRDvMVhHxMyObvTXK4AenZlX"
    "w9Ofh3RwgB5QdQ77ilank8vZdHJufhpefxpdQ7OXtBH+dbcXkn+s3ZAs4fRGLde/gwMDo+p0"
    "2/Zh1zro9/bn8/6BtZgvSHvu7DtO+7C76BBrcGR1jhYHh9bBfs9xBp2efTS3Fo7T6wwG7cXd"
    "XqPYTxR47EB7gf2lRbcx7axzcHhwsD+Yt61epwto2vvthX10YNtwPNqL9kEXX88P+4t2nyz6"
    "8+78sHvU7pF+xx70BodtW9HZEigaAjNy/0lPdov2GD/HrEOn3e/MnSPrcLFoW/1Bfw5L2LW6"
    "c6fd6VudxT7pWDDB/f3F3JkvFvsHMGtncdSH2fW78NJRzk6iYra33qE936eEsq3Dnm0dHB3O"
    "u4P+YX8AqNrkyJ4P9juLw95+x24PFpazfzhY2L0ugYbzfat7qOjN9aPYArYZAmOEc9VabVhX"
    "/cF8bkE3nf0OObL6i/3+/lF3f7990O8eDQ6crtXv9ebk4NBeHPT2DxdOd9CdDwb7hz0gZX/Q"
    "7agmxrowl5bvLkgUS+s273V7Tnd+dNB1DruwLjCHgw6ZO4NFb+D02z3octE/aDtz0l30nU6v"
    "fwT/6x70O4P54cI+UPUWPVrQQNXZ4eBoMe/uW/ND58juLboDxzlqH5FBewCEtfoHTv+oa7ed"
    "wzlsWLvXc6wjcnTYG3QOncPDznzgbN8kZkhs4q7STmtZAAZ00LWsHky614bd0O5b84P+AP4P"
    "PwyO9sm8PTggCwKUPuz1+0cw5v5Bj8AmXfTm+/32weBIPsv4V10a1mv6o4JXIH+s3e29+2I9"
    "PHjkHcpd1394Zz1wlrFqm6uuKfibmfI3ykHqEnP4PD4bXQKL+3V8VYZXzTfJkwuS3ibANVsg"
    "AzKYp6M/38CIzzirQ35Ty1M9GaBLFYZ4A3RH/s/JXlgle+1YphO6TyQ0PdcHtWMLwNL13aUF"
    "OkXoxrCqAOATbwtMsGKahpJ80XoJqsxGBztf+45XOCCFZo9raCGGsczuR4mEw/PzyV/Ox9fI"
    "vEeXn8fTyeXFiDL4AiXf34zPz8yz4Wyk6O7jeMblluIlCI6P5yPz59H0cnRugnwyZ79cjfQN"
    "UYqAOBxNzcvhhaodjON8/H46nP4Ccmn2SdWi/DWqsOaniRI5fVeAk2h2ClsOaDQenssko2NV"
    "bsHh5ezTdHIFysLwagxU+EXR6fAv1+bw9HR0fY0NQKvQtLkeQe8zqamq3a83UxD/V6PL4bik"
    "y4+TCRJb3+DTB3M2+Xl0qXp18/Hj+PKj+WEI5/rTzXttw2TpVR1sHSI2mMLhhp2gaiStycXw"
    "r+OLmwvzdHg1w9mffhpOh6CSTHFFur12uy2tH+zT4eWZiVoSIEeaTi7PsCGoIUU9anx5PYNj"
    "omo+kNGmPwVRi/hPbhj4t5SKSKDJhw/nsKXv9u6ZSnmnaz+bDi+vP0ymoAteV4e6Gl+ZlxMY"
    "69norxVa/zL7NLm8nNxco5Y7U+NX/eSQhWFbfuC7NrA8ZD21lbXxAss5NoL534kd143mn4wo"
    "Do+zaxkSMIN8aky0nPVyFQm4hkH8CC0kK7Jd92QWrknDiMjKAjMuCKOTGq74XgMGdwy8H14B"
    "PwMmu4lo0/r24XIZj8ZDDU2AYyqBNMN03AfgqUALbk+1GHStnm321Y0fqT3RAnvThyGGcxic"
    "YUUAh9z5uCjKgcsb9uPa/2K4vuGCJlDzrOXcsY45SCskllPrtLs94ycD/4G5AlJAq0CWDrW1"
    "XqG5U6Oo60qa84aP5Jn9VKtAtDnYwA5xak+WtybHSCgNwdyF4QEFaLu68ccTQ38QFfPgI6TQ"
    "yrHTN7fNEqT32ycDOiUY9SBUfSfHm63w4enY8NwovoWp3ef4z0+531ExBbv/GFYQt4iGjeSA"
    "4NAdwxLYtIcGUvDe+E/jMvAJoMB/ZH6GNE7bsgN1n6MbqOJhTBzTwjEIU7flB19rYPzWW24U"
    "oEJhxfk9G4cbxQoAXVYeAXyALLXIW0CymnrbIcka6lf2I7G/nHywvIjoWlgr6g1h3hN22NUt"
    "Y/Jc+h5UwsABvfHkbm8dL5qDgvBIGoYhshE0mUDZsom+of90Av/pxsPW/oT/q2hV127vFzVK"
    "MHuBlqj5l9AUx41I0FOFTZP1aqXPtaBR7MBYEUyc5xScvauXwQLptLDwTg+LRHJM3nXZfsBu"
    "xG7GtulvWoBkFByk9AQosLzmjuezTVayM6o1Yys8el6Bge0gT4c2ipPDCFjU95L99GxzIrcc"
    "gstUS7eqYlfW1ViAv7oRtb/BBqqlOEE4bGISaaAIkNyQWqOPlMqSStuWLe/WmUGb7z4zgN5h"
    "ZjjKnWb2LziQORa+7QDufux2OmwlHPN/zFmbXI9wj+hP1m+3SriTdliaxd3eCzrqcT/WW6bp"
    "W0timq/Hxgs8eEV19d+cQ+o0qq8haLBM/cc5UwWxYagsASR8bolRbwbmUnA5vTMQV7Fti/WG"
    "GkFNbXvUGwp9oIKWC0ZGBMpIZIbEI1ZEajkFDJU19eCFE4vE9rsganL4AhsCHoc3WXQWbsRs"
    "kHrJjn/NK8HR2osL6uNJoSEaF54LKiXYFrQzNCYYxRSKkopHt6KV58aII1KOEGZytwcQdD7Q"
    "C7bUGCZ24Meun1fo8Q+YbQ2m0sMcEAPrtUYxN4yl9Ux/P+koWTfS4hZQIAEoEhABobuq1fm/"
    "P97t/ai2gBhsBcuWGQmJq1De3E8kjNBnB0wnkkyGSio7vYt4Yjo2vQxrfX107UeKPjfiYB6R"
    "8Ik6iI+LeOmFyh49JnhBCf+iYYwrjr+Lbl4LmzAZgBvRBVScylznt3d7fMbMNyDbULcCXcP4"
    "SSbLvZr6EtbtS7AMnLVHFCtQzTZaERsGm73RbC1cWFN8pSK44+LumbObIj4b2mHWSqtiS6lQ"
    "ZUYjrmBb/KVqPFxIKoCuLPuL9UAug/gDyh8qRKuPojgJvZi922PLkGwyVZMgdB/Y/R0lE+wy"
    "Sn3YYfR3qqjhkxZrqMShGisVWYrnOQQVpBRgsQPmx59Ttz5ID8dFcRgxTk9PcWFPFU5vEMSR"
    "WhsWYmAdhe+8AMTSO//JdVzrHfTY76kVgCIMju2tEO+ieD2PKsC583fPg77Z7zWB8a6fmw/+"
    "uhLUtkFVQ5zb5BEheMgI46HIFknBXyGxDcFwi0sFoLf3RVmIS4ZSii6dWpohG8TXLfIMqNVC"
    "r1ya0Y6sr9gPOiVBH8TIjBb+BUp7WKODeAe7nG+/VhT8BJSp63qSVQvAqzGAfFTcPPefTJpA"
    "NwimN7Gk5jhOpLu6bflcxaK1LMeppTg1/cpL17JWK1JwvMl/XvSv+BUel29pv41tIGxREYiq"
    "Q2KNt8KBkhZtlnjvl8CmjyrAC8loijGLFWrxN7V6HZclMyrGK0tMFMkYgNmb1D5OxgcWQUx1"
    "IBNffgty27OiyF2Aeh272ivpIhi9L5tNJuc/j2fm9ezmPWiJVF1kvOkd/urLG5GO6G5vOhqe"
    "m5OpeTYdfx5NzQsMLMlfV+f/lK3Aq/rVVqUkuj3udAcV/Mh4452/TWaCJF6DxVVkTw2wFAMv"
    "L0/S2DVqMEbmCvYGoS5lzkeKE8GGlOHQH4CcpXeQQP30uqf1ALxVoWjkfrU8L/iKXBY2L4dc"
    "skG9MB1MhTEzJM1Vcm5ZmIA0o6Wb0yqLk77V8CFZhQZtmCJsAkK8gqHuIPmR1kxuNv+xJuGm"
    "+bBan4CCSJ4bVNXhV/9C6ViSZRBuWnEQW14pMmZQn9jRU8MPHsH8IiH8sPZdZEMKuPvS1ZiD"
    "NWnGQWg/KqyAKrooD3KjKFT7OdEWyZNrk0ghR4WIo7ShwtTyH0iNYmxRecZgWbxkTS/WQozJ"
    "jF3aiQQNm8jkGNImNdqbRqzwsX4HiUJ7QQbHFn5bc2FwqUePb/m4t7PYYLlax0Aza2XNXbB1"
    "aSQVHruaGnnakHexvQ+6V022cSVZkdC4JTf4Zn6a3a3FiD+J6HRLEmern1IyBBhRTHEcTVPv"
    "9UIFf752PSeF41CUqFpAbnGm5ozoE7012zq0nizXs+ZefnuA0pC8qundgPIJymHIHq4tGOgK"
    "8x8r+2FH9B+0DrWe2N0XttyXSR1PNPqazrbgOtU7Qa0FMSl01jUeUvjKXkxx4wkrG2pMuoIP"
    "CFYaQz5BGgGXT30iqj4VsA/fAox7+a2w/tO39LyKn63oWwYOpsTCfRAYVgrQHB8RIHlfU07a"
    "p5i5rJe7Ej3lMBesf7w4L/MN5DxEiYhMjwNVgaTTl79r70GzVeAFDxv1LpNuvsSxpVJAwVxh"
    "XDTEgrerGycnRlfdDHQ4jVSU+nNjsmxQpUIjZhHT3d6sx/R2PGMIwmfMRGGD3bnVW2sQxWGt"
    "BJEEqpJ9dDa3hw3j4F6Ng2oggAOHIqi97aYvr0uhUEjDP8X1CK4naOcKgmlUYT4NOQyyLshe"
    "0l6OjCwfqOBMqejIO4wZL6rLblw6gsIhyTXKbWim4jITiPoRKCGKJ0EcscR/0eqkm0IcPD5N"
    "cYkmtkb+2iSKaE8q2wYGqbOJ1Ptcs6LqxtJZVDcokl3dLkc2daPUvOF0ke8h6W5vl+4BFrCr"
    "lrZAZPuRLC3ZU5rNVtE4WZPQYpfK6Ez2ihIElMU54a3v9q7aSgtGNBPacRLsOb7ARBEMjUpC"
    "KKmtOvtFg4eHBb31wpLekcbriI92eH09OmM+CL7tuMPhw3B8PjrTjMEhtiuICkjOhzOMwUxG"
    "LkLPlIhlgl5+Hp6PdZ2wHDFzabk+6N/hA6FyzqWz1uUBqSm/iR/ZWF8qaNHCa9FiYGL/lKiV"
    "Lp4oZGKJKyiPI9uiBBVLrBM6crSJWukTld6mnq9ILiubMeAGSZEZLHtUMrrk5lYCSm6D9TaD"
    "ZT+6fhaKPyuBSi+amfMuf/lcmRoagUNjItSv1McuZbpS0xwDxpgzDXNWIk0ZIHXZJr8pG6cK"
    "FVXrk9/0jZm1Ro+6xj2Uv0FEg5h2oFsYBQBN5VBD3KtJKYwKEzXhGMRplBj2yTs172Kpnia9"
    "pARJG1pgtaO4pZRP00BbcoMaMz/VI5R14uNEr1Y3LIh8Nur8U/XAMdwzKj2R6c4BwszFfioT"
    "+NrDw9KFTJ3ox+2jfqU3ndeYtSNpB2iSpr+WxtcwfYEuCE0RyvgBigqFfgwi0Yite6JboBTM"
    "qhtaFOkRM6O1bROgq5M9elpNpDLDma8dwFG+1jRV28RwHNqurR0wS+Y2aSTTlqYMJyY7kmhb"
    "2yQH3IxDC/2k6P6qBsrTyiu2Js8xCX3YOxG6IXVtX/X31ZnIFFT3GlyfqHAF0TGLCWpvuIPA"
    "6UYryybqQCwQuB0zaZO/lEletJZfHBfvt0LMQ+W5G/TiyQy+5PMzqOue6j/QZ9r/O0XOnUkV"
    "y5ad7zhVHCqiyCNg/csBZardzhaYRvnb67HvxrW1H7kPPr1Djev/cQd/fHyLelztKXCduvEi"
    "VpRDtOv/AUuODZWq4NbwdY1pqDEICwE/SXsejqGJ/EcPJwuOKvPviXjLW9bb/W8Sc4l9G18t"
    "ZtMu0AFYAlYpvDLvFYVWVchUfmdF760E9Wm8Xo1tujp1T/3FazS5kdz0mAinPwf4L3Nnii1e"
    "v9/q6ABGZ7N9rLasEWF2/Bpj3fgD/oB7FX4qg6KLoTbxCyEeHrAfwiQkF3DsDkh7yanZgHHw"
    "hfh67xB9jU6JlBotHtcIY66x4RoY4pg+rUtPWQiiPoJC9n8w74dmQK/l/s2UDKCbZG7aaerd"
    "zeV0dD05/4wWZuE4a4ip2K7l3dR0B0Zxh68liOWj/pm53KceOtlVpxlwWVB/9YCAgufYMYuR"
    "siJisiSKLy358iZwUapAUJslF26JHEx5c/ZsFZUz6k5K2UFp/HBKgq3uc0fynDsqrlNXXwOL"
    "8GLm+nPK+cm2aGLF0RKuRdoPc0Gf/Cl9VBK1pFqHQnzxn3IBxred+zR8mHGA2/a9vpN5SKwv"
    "Cjme20IF8pcTt8wtWnGbqN2QhXFpncE6iB07pB7k6qceQZTrph3nv4OftFPRT0q5Ledy5+PL"
    "n+Gfz+Pr8fvx+b+Pj7RclGQnR6sCDU9npuhXyxHydpBCSpzeTKcY6pRxNuc7EtNQCI9S/00S"
    "t5eocCWt+UmRmvMnavqrDwd1iqpfaVw2WgHPfDfa1xpXkSNNI/1N2Vh1YmnigeJ5JQSVIw7V"
    "fEtkN2gYiR6IZRHquX3VoEZlz28Ib6xX3ad5Lo0kyz9TAqLqwJzh5nLN68+ZKxIiy2D+olu1"
    "Z1P2DOOFeeg6JDKtFcjMUrgq3kF+aIQfLeu/qij/tDZg5iQwr+CCD7lUWOmt29xiZ3Fuk2Yl"
    "cUToMMjOvWRrlkn2bxXkZRG2d3trH1Y9DDC6WbGbJP+uJrjr7a7FZKPE0OOKNe5suXCiQ5Lb"
    "4/EtJR49t/r99Og6sOAAGAu2vc1v+f+1L/Rf499MwnLSgoTMu4lZEfk4ajmeR+tUyAbfq20b"
    "dlb9fGky11+hUVNvhTSnQ1dXsa7MLxG5A+ggZWEiciZoklFQquTy4ipyQMbvToyOKoHUcmHz"
    "ThkPoplhGhaywPOxoowPToNlx97GwM3/opvda8NIAkVecqN5rWCdC7doApUxs3T74MnyaGt5"
    "H6Q/SiV8tmUEirGbmNsSBp6JhX1IVDGnlmVYJgSDLaKpdNlCx4fa1uXmaDp6ZaKzNlvY4Psz"
    "s2s01rBiF8B6Sz1zIhiIGnnf0gX1y3/APtY+xlges2j/V2VBBSlgKF9UqaR1jvK3iB/prY+j"
    "wfQF0RXsd0H+b520uBnEm7UF4lPNNduHKLIHw6WVqyhHr9Uya6mrIlkvTcLOe09haNw9mqBh"
    "cpu/0KWWyyU0WEseSFiRSWQIRVW29TKdthMQJiCRhhZsf91oKBYznYTibh5PiCbDgDsNBXLV"
    "hlAHMLLgt2yEIjNKUk8UFa1YsEtMFpX33PDvlXw3MynKetXVcKuROrspVyyZ16Dx3dKOLPZb"
    "PIsgH6naKdG7XeRd+HIjkVWfBpnbQxtO3OrnTb2L+AAiGANl50qeEhIP1McndJbS9plFrKsc"
    "bowZmHDocjDsGG6BwhS5HJiUUqcW5TVdJmaGdGImVKDoEiPDAsGl+bwVEibQoDeTCmH8fRYR"
    "dpvnRIYVktLFVIk6QZbfQtzx8iA6SSeG9qpbeFV+pSSQ2APAWJCG2VaP1jdPoCi6ygdfYBF/"
    "AL2xevq/VA5cTmRNH+u8aLKkRyiNEqCEFlKHjnmTJsUgSxTMsKEdrQoqy0iVsBlCSeB5Au5W"
    "fYArh+cgKWbT8WxyaV6fTsdXWFY3RLkk683f9J2ACuX9lfX78zX5dymQr04tFE9Z6Jz6ccuz"
    "/Ic1ij0rMmKvnIb/h8P83Y2ztoHlOLzKco4xP5urOO+w3SieseqDihf+ejnHkKuFSVhsa37Z"
    "359PTn82r8e/jo5h/C3Y1BHodysZT55ZBYtFRGj9CABYhcFDaGH2cM16dqOTdt34SUJq/AEb"
    "WSzRst2Q3tTzemn0BVVojvuPipHnSMP6R2ZSo2SCrjh0g2I7wb9yvWwkoE1VIACIYGuQWkpl"
    "GewZftmooVU/FeeFZXfb3V7a5DlJK+WEU6wiTx05EWGT8AAjK08Y3AJmGO93paFsEpyLtefV"
    "VCjrDaPbau+MmVElQY/uqY3puV9ITc56fQhdeivIj4DtuE/KaYHUqTckuPRo3CKK+/wByR8O"
    "5cF4+6E4gfGoyzNLiY3Rxrcfw8AH2Sm7PxIj+4TtkfRNcklK3bacbv9YWx7fYqmBLhtVQnQq"
    "PkFBUZg7ZJkKkB1zTAXYbhmmIvzY1BXHKaknpA9dTrDywUj4QA22YpDurAEaThIlcEejd3Qb"
    "PmlybM9qZ/f2vNm358smkBUyudtF4G/J2m7X1fgyt42qmL4CFM3J1IHwhM0ilEIy0G8bbTnU"
    "tJTJ2gNlDJ2ELF8Bj2Hx8weJq1ufAJz1c5e1K3dyS1rXKgSDp/bdy4lLDCQxT3Da+ZJQVHG/"
    "pjcgo2c3ru3Lcoz+T0SXVFYUmYZovp9MZtez6fBKpSsK7c2Vk91KPppE0wfepuXBSuGGxq5E"
    "USLM3cFY0NYqWNU69brKrc6XwZSLGVWB01ZvR8MHVT1abyS3DIz3oDClaRIX9NcZZpUr6p2w"
    "xglfAiiwtf4of7cpWq8wCwGWu8k8g3/K+7e5zcSro5UOHifNk1Vu0ckQE3sNitESxB6rqJfM"
    "rPC2rsMChleox5J7K2MB/C0arKykuO0Ry6cSVq4Bgj4kVqsRo75gHKviFQw/I6xKvCKEiSEW"
    "BTtYkXhVpJOq7JS8gxik7hoHhAbbrdCcQcGOkLYvBolKv4oamTxsvYCLu+bEFqbhjgpXR0gr"
    "jL2w1WtyoIjxFhTd0qNX3bUTwyFGkx9L0eObH524VkoocFyVuluWJsMG0fSXCFgXNUA5jmM1"
    "ZteH/RijGZOHz+1vnMntMW5mDpp9jZwDtyarpJVyFzk+OlX25BblDnhZ9SS2iHhO5Uihvwa/"
    "W84673fh9sjQOCFEmIDjhvR6FmvfIHHE+kbSHRetfYe/3b/pBvRFX26u7GI0fzmaHR69+GAp"
    "RnjVQS9I7/Z+wnKAZRW/33Qruj24unDJWCGTpmtmv9z0lkpeaUzebt9ikD+3x246NfVeyyKJ"
    "xUfZ3hK7zCIS3waKeasOy6/ZEXQBmj3qZiiut0ZKpyVftjYVsTe04qImoYmfO960kBiU2dsn"
    "WYz0ji/L5FXUjGxgAdr+c58JYxlO+CG9KtW8JJfwiTpGogxG2mOsdHLJ1braES6TY1vBTOXV"
    "ApfPHI8huN7GsDxk0htDFGxUfY5AXhtFyplqyPxsCNmhKSS3LRed13ZblmQarfCTdCVl4YAF"
    "eBY1eUoawYriCJqATNTaadIbmBIoPsVSvH7Q5JXOShrxT0k2hdO+pGnge5vm3PVBcJ0cQ+fH"
    "pa3Zmmub0NQoeWnrZdX1YB5YX/ZLVIovdzHPrnvL6vA3w13wab66Wd9e3k/elNnqishAa2lt"
    "xXolOGpJic948c/YMTfN7uD8S3c7gFf53NgW8CrfH8uj2CEDTz7/Wz7os+UrbyUfC1JMrdpn"
    "gVCRz8xJGaL5uxOjvQuLXbk+GnGCxfIeGNdHwUuckkCXvBSTU2S1l1lJ0uCWzy/kE3V2PQAK"
    "uF0OwDbwLQdAAb7rDlajoCHmNO/jfGSejT6PT0fXAk27OpoK3xcsR1D5g4NpjmGazFnVnFFy"
    "lkeCu8wxPUdSuYoVtApf+6zrxrQxqSEbpfYimlCyFZOM/V5zLtMxaQ5ftidhQMuAlTdRYWI0"
    "Lo5ZxBFZtf4euH4t219dd253yU/WR5dXUom46LzWik7ewC5tUOL6LIGqrDWIxjJX07W93yIj"
    "qn3pT5YNivWuLBvk9fwukoGbHsaMmh4GuyA0+PfOVLIhpR//xFlKyC1pn3kviOQuY3d4Uhgl"
    "/xRWlbA0Bv6GuLTc1PntqygjgPYkM1s14UyppUtdS2GkCVet6VwsCJGsoXyfkpTnQwtGG/KV"
    "QaCun/i78vqJNNOX1nLkTjvGFiW0mZux7YUdv4X8nEfBmFinaVxTySZMrn+LlKpeyDZBQq+d"
    "FO5YyTPB7xOylWkVhyPxT9Cr6VxFWp2i9T8yb7VbMW/1Ynw5vhieCzWQfc+7Wr6q9OtvnJ3K"
    "skdB2T/9WUzkajq+GM/Gn0ffNUE108X4EoQICHlU9qqnpSp9ODg/5YvSlEvZHmCVz2ULRAma"
    "dZlR54+IBJSf1cuAM7JaQG8V4Am4Qq8sKf2mVvm+QUVVK4lZh5RwhymijnE/aNIj2dHiMpjX"
    "Z5+X5g+zBvLHtEQRQS6i1HmfEmNDAPl39UHMVNpOf3trkqe0U5PSbMIF6Wi3cqXMSdXWrpxH"
    "mfMq5LMoKxjob8mhFF5gpoOJ9a886Iyqnx/yVr3xOww4HxRSNlZZeyqMVXqp14/0ulH9u+aX"
    "5jZSMW+0ZLNsyRrldJNTnUvyU7Xr+7+5qb99biosBspCk8HVhA4lPn6a6ErJAzcITaGUVP+G"
    "YVlc/m+hH4qfK2iI6W//+hIml5MZlpc2z25G5myCKtxkSst+3ExHFRTE7Opo5oZNYhICDwQO"
    "uCN0Fb4jn///PdzffrjLP1f8uF5ayemN1ksY5kZxga37bDGtZqW+TLzb+70xXIfWRxZIZqBn"
    "1+h0W0fGVbt51TWueI1o4yw5ipotqny8uNtrSqDG+OzY+NtL5hS//k0Pec2KbmL8l0GLjiO0"
    "rtp4GaIJ8B2QggY7hoiFU/H2R/box/sy8Bk/SUbKEyUMhXOmRaahEizC740rqm0wARpVhVZ8"
    "QJJrLb4hxsdZHeC8V1fdS13BdLIvtPntj4J7wmyM//tf/508TwhGn/5NPM5Ovq7YgS2sTrrL"
    "59JK62sCya5BxUdXUuMN4E3jApmDQRkOHNfSpn+hLId9wXxrY4ZX8IJtrd8LtmOkbKcy8CVj"
    "PJXbn9IYS5gJht4fG6LuSQnEiFcMNmjF4HL85Yt1iZEBnuUuo7ehmD2CVpkqBGmCNy3sYlBW"
    "b6CF6foPNK/Cc203NrgTYzibXRpMchg6Hwn/YP16hSmpCxLCyhBUhsjCfW7aFigwRkxjzmmi"
    "MRb9jo05ebSeQKI2DKzJ3dzaA4aM4FnANCRiYbi3YwzfvX93apDFAlafJvysvGBD/e4GO8/O"
    "2qaWZQLc2pWE5Z/sqynDfmSh01qih3Nb9WF6oyqufny1f2SX8sHlQnG+9h2PJLULajrptySY"
    "MBBpvrcrPgE5Hf35BqZ/xquIXGtyzrH1yQlVkjK988oJO3+Jl0fqKhZAXWiDT6b8Q4IvZXdb"
    "ySdp1bqudBh4UvpxMUm4HG7bh1410K9bb5mkmhbfz8bgCyk8ynJUvb3udI+aq3Zz1W2Kj2U0"
    "yRM6g2zSfNLVTHyD1fLdvq7C9wfC8R/LDUp2lHDz1nR7upHQvcLRZF8yEDQy/4l3appzCcdp"
    "9Bk/UgOT+3V8VRacp842F70YAJ2LxgPLRcZdzDTHqhoGjA73dOtXd/UB97YMgtdJX3Hy6P/A"
    "1ArYUScJwPjKPBt9OB/ORmd1vLqxQvvRfVIlEuzIY97CE2hVUjYAxqHpIW3gMwQ4yWWU6JaO"
    "ZVnoFys/nt0XTJLe/DpTF00J61eo5lRYw8KASgMsc3GvbTNxorTN5I5N/Z3kkiNTaC0Epjg7"
    "SUfaoYhsLTEM1R1gx0xu33K+HJ0aoKnoqi10uqWKK4Or+LUo1VVod/cZdEtGUv0+71tHXrDu"
    "aHSRCl+lovzJUjbwx3Tjqb+OoS890tHvljfReyvN30L37MVlpbK3jcplRvWL8411djUrlyFr"
    "A39MV6+YkVF9vOUL0javumbyTTdp25Xf94o7XzHIkqqc33D1q9XXC3yyuLkLjLJTZJQFNLlU"
    "hORqJ4upu43lMoeI2iOmdXlmPMuCq9/Kz+9LPgYi3L8ypHh6X/KNZMlzK0Gmz+8rXidVm1bn"
    "zdPqvHlanX/9tLpvnlb3zdPqvnVaOSOVu/F+67ib72ahfMerFL3qIIXhKFjuSSXGmo3cMU/P"
    "J9e71GVX3r4UHuqvptLQikhDhmBlAg9cuCHY1Tw4oqSGcf42JltCu1299Dao7hFou/QjW67/"
    "kESlbPR4KtztVLzXqXqn88b7nOp3OSjSmDPVRGeq/NFJ3Vebsh8VXEfln52v+rU2WKW1l0hE"
    "LC6TVKUXt/RYwgzYyLb+fOjQRN9H6dlKvp5qCh+r6Bt782m/c8v+QoesDUZ786ksjV+jixA+"
    "kdQBgVELvFT/JjXtktOy/TCX+UsCGrpp2iliibFyvp1oJfx3pXajut9TNiz4PFXfC896X/Q6"
    "EJYhUSTdW37gwwby+Cx39zCmDDq9gtomPL+RZ0qHRpq8CNqT/Tr1ytCm2gFaHVnFG+0db7V3"
    "vdn+xtvt3bhiNT/uFmcUc0Tt4S/38NfrD8Y9vNkTJaMwLOEHrBhm28QjtDgOPNr7eHVDEfDP"
    "rfJWhWYs8X9GIs+a9ViPe4iVXTVH7KsQ7GkATCwcL60H8plpV2OH1h/yPNbAjT6u1iMf000c"
    "Wj0K5C5/MfaRdZM4fbugbJe+FqUEcTjsk9Q411c2eGrRAMO3kwk4bgT8ZUODUxDkioIY+3ta"
    "bOyFaM8e7qd9CAjT9RdB0k22OcfB1Up8vt/qdPfYciCaPX/OtDR415N/R0uYkvrgh9cf/h9Q"
    "SwMEFAAAAAgAAAAhANbP6/z6AQAAfAQAABQAAABidW5kbGVfbWFuaWZlc3QuanNvbq2S3W7c"
    "IBCF38XX6zX/4LxKVaEBhiyNbRwbb7WN8u7F2TbaJm2lRr1DnOHMnG94amApKYIva3P36anJ"
    "W5m3YicYsblrYFvgHgp+hYv1G2W9nYmdmZ0HKDEvow0J7qe8luTtmR7TfJlcc2gWnPOaSl4u"
    "doZyqkZTLuhyfli7j1rmYR/oRvxpWcX1BEyqKnvAGCMVykhC0ActMfhAmOGOB0MUSO20wh4I"
    "8IDecWqiF8Twqjkd5e6VvqF1l4KVh1CEy+fDWyp5LilP1v9l8gUfN1zL8cuap98CCVCgwzMM"
    "a+dw8qcRlocOp3Na8jTiVNrHDYYUk4e9V3um3Qe6vmP2o+AWWaSgdeipExFUpUSNRqcYSqYh"
    "cDBEewApUQdUoWcu8GhIFF76vpfkHTJayb8n9i9LT+M84I7gJfn+B/4I8ZXc2g3Zw2DB+e7/"
    "9Lqie6Mv6PMSbulp4jVzDPVOj9WPR6nsCangOHMGGUTDHBpHg+8148g4EZEySZC7Xnnxll6v"
    "zPPnQ+O2KQxoU/iVXfuSp51JO7N2zdvisb2WtmdWrW6SXVU7QprsiMs9Wp/HMZV94yISgwGE"
    "E0FooSWtJyVURMmVEsGANh7US05/whHsGZe15q9v2ZEcyS5c7W9W8mrvOUQlDDhDhGI9OvCC"
    "Roa9cjT6CIbWGx9o8/wdUEsDBBQAAAAIAAAAIQBMOYTRUAMAADoHAAAvAAAAb3B0aW9uX2Nf"
    "cDBfcDJfcGxhdGZvcm1fZGlhZ25vc3RpY19yZXF1ZXN0Lmpzb26VVWFz4jYQ/S/+HHKQciTN"
    "Nx9wjCbGMMZJ56bT2RHyEnSxJFeSSehN/ntXBhzSuL3pJ9vap7dPu6vnH9Eatdgqbp/AW/4d"
    "hTd2Dxb/rNF5BxVaJb3HIrrtX0TCYoHaS16+i2zoGylaO28UWii45x1x3EnaLRD+khVorjC6"
    "jfhjT9SDq197Vb9XXfWqkvuNsap3wvZ2g0uCR7T7xaPVvARXoT7I2XBZYgHaeFwb8/QzzkJS"
    "5LCHaIlyQ69u7zwqULXnXhoNTpgqkNzFs1kyhd8W2R1LZzBh2XScL7JvsEiTb7R3KwsSSIXy"
    "VmJXNUq5FnXBwe1VKfVTB0LxF6lqBQ6do9Quuh3QoinO0k9YPEsXq5yNo0OohNLwoousCf5L"
    "3zT6Z0Md5kJQro7dmqoLj9wjUAlCiWobVFQWHdodQts6rgsQJSfBmz2cKtvu+AdRRThikaoq"
    "UdHcAL5UpRTS06hJTwBOInRT9jUXT6GtxPA/uqmN81IcmmmqhkhAgUKGelIxhLFUK+63xNPO"
    "uftUGkFzxNfiE68tD2qf+R6aLNDS2JqUKYS3PG/MNJTfndGUtbJmjXTI338009hCKOFb74Cl"
    "D3HCJuF0h0Md28vm8WwKcTqB7D7N2ZxWJlN6ycOEheqd8y2TOP+6yOYtBsbxMr/PppOTDpBF"
    "wPWj14uPcsb3WUb74F3qhKV30wzGizTP4nEOX2OWTM90ju8nMUwy9kCgI/aBrdgXlnRr7MC3"
    "3Mt4tfqgdfCfWld5PL6DPGP5IqUijhfzZZyzL8n0TeGcpXSU5ASijOk06VTWQbnM2JzlpLZb"
    "3FX0+sdFdLxTh7WzgfnpWB43HsYzfEiyTzC1r2ofRiZq749sXNUH362M9afhavyjsHJHnho8"
    "hB7vAUpqqWiUj9eJALoxgXNMO9BttrOJdrWiK7E/Yde1LkoExbXckPLT8ramlROtKqJQlOPt"
    "OJyGKO3hz/G+RLArS3W8WM9bxHJraofQFMSJLSoOdLZjhwaX/ct+CJjaiiBCaqC/ySOCMIr8"
    "iiCb4aZ/gwUfrofF8Hp4/XlAb6PhaIOffxmNhsUNv74RfBRshH5FVWNm0lLvWkvztia7O5o+"
    "WbSl0xN/te8wxWCZVHLnufUfwq9/A1BLAQIUAxQAAAAIAAAAIQBdBJnigQMAALAHAABCAAAA"
    "AAAAAAAAAACkgQAAAABhdXJhZ2F0ZXdheV9jdTEyOV9wMF9wMl9wbGF0Zm9ybV9kaWFnbm9z"
    "dGljX2ltcGxlbWVudGF0aW9uX3YxLmpzb25QSwECFAMUAAAACAAAACEA4v+yscYlAADTswAA"
    "NAAAAAAAAAAAAAAApIHhAwAAYXVyYWdhdGV3YXlfY3UxMjlfcDBfcDJfcGxhdGZvcm1fZGlh"
    "Z25vc3RpY192MS5pcHluYlBLAQIUAxQAAAAIAAAAIQDWz+v8+gEAAHwEAAAUAAAAAAAAAAAA"
    "AACkgfkpAABidW5kbGVfbWFuaWZlc3QuanNvblBLAQIUAxQAAAAIAAAAIQBMOYTRUAMAADoH"
    "AAAvAAAAAAAAAAAAAACkgSUsAABvcHRpb25fY19wMF9wMl9wbGF0Zm9ybV9kaWFnbm9zdGlj"
    "X3JlcXVlc3QuanNvblBLBQYAAAAABAAEAHEBAADCLwAAAAA="
)


def canonical_json(payload: object) -> str:
    return json.dumps(
        payload,
        ensure_ascii=True,
        separators=(",", ":"),
        sort_keys=True,
    )


def sha256_bytes(payload: bytes) -> str:
    return hashlib.sha256(payload).hexdigest()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def safe_basename(value: str) -> str:
    path = PurePosixPath(value)
    if (
        path.is_absolute()
        or len(path.parts) != 1
        or ".." in path.parts
        or "\\" in value
        or value in {".", ".."}
    ):
        raise RuntimeError(f"unsafe source bundle member: {value}")
    return value


def validate_member(member: zipfile.ZipInfo) -> str:
    name = safe_basename(member.filename)
    mode = member.external_attr >> 16
    if member.is_dir() or stat.S_ISLNK(mode):
        raise RuntimeError(f"unsafe source bundle entry: {name}")
    return name


def require_clean_output_paths() -> None:
    if OUTPUT_ROOT.exists():
        raise RuntimeError(f"output directory already exists: {OUTPUT_ROOT}")
    if STAGING_ROOT.exists():
        raise RuntimeError(f"staging directory already exists: {STAGING_ROOT}")


def main() -> None:
    require_clean_output_paths()
    bundle_bytes = base64.b64decode(SOURCE_BUNDLE_B64, validate=True)
    if sha256_bytes(bundle_bytes) != EXPECTED_SOURCE_BUNDLE_SHA256:
        raise RuntimeError("embedded source bundle identity drifted")

    with zipfile.ZipFile(io.BytesIO(bundle_bytes)) as archive:
        members = archive.infolist()
        names = [validate_member(member) for member in members]
        if len(names) != len(set(names)):
            raise RuntimeError("source bundle contains duplicate members")
        if BUNDLE_MANIFEST_NAME not in names:
            raise RuntimeError("source bundle manifest is missing")
        manifest_bytes = archive.read(BUNDLE_MANIFEST_NAME)
        if sha256_bytes(manifest_bytes) != EXPECTED_BUNDLE_MANIFEST_SHA256:
            raise RuntimeError("source bundle manifest identity drifted")
        manifest = json.loads(manifest_bytes.decode("utf-8"))
        if not isinstance(manifest, dict):
            raise RuntimeError("source bundle manifest must be one object")
        if (
            manifest.get("source_repository_commit")
            != EXPECTED_SOURCE_REPOSITORY_COMMIT
        ):
            raise RuntimeError("source repository commit drifted")
        raw_artifacts = manifest.get("artifacts")
        if not isinstance(raw_artifacts, list) or len(raw_artifacts) != 3:
            raise RuntimeError("source bundle artifact count drifted")

        inventory: list[dict[str, object]] = []
        expected_names = {BUNDLE_MANIFEST_NAME}
        source_payloads: dict[str, bytes] = {}
        for raw in raw_artifacts:
            if not isinstance(raw, dict):
                raise RuntimeError("source artifact manifest entry is invalid")
            name = safe_basename(str(raw.get("output_name", "")))
            expected_names.add(name)
            payload = archive.read(name)
            expected_sha256 = raw.get("sha256")
            expected_size = raw.get("size_bytes")
            if not isinstance(expected_sha256, str):
                raise RuntimeError("source artifact SHA-256 is invalid")
            if not isinstance(expected_size, int):
                raise RuntimeError("source artifact size is invalid")
            if len(payload) != expected_size:
                raise RuntimeError(f"source artifact size drifted: {name}")
            if sha256_bytes(payload) != expected_sha256:
                raise RuntimeError(f"source artifact identity drifted: {name}")
            source_payloads[name] = payload
            inventory.append(
                {
                    "role": raw.get("role"),
                    "path": name,
                    "sha256": expected_sha256,
                    "size_bytes": expected_size,
                }
            )

        if set(names) != expected_names:
            raise RuntimeError("source bundle member set drifted")

    inventory_bytes = canonical_json(inventory).encode("utf-8")
    if sha256_bytes(inventory_bytes) != EXPECTED_SOURCE_INVENTORY_SHA256:
        raise RuntimeError("source inventory identity drifted")

    STAGING_ROOT.mkdir(parents=False, exist_ok=False)
    try:
        for name, payload in source_payloads.items():
            (STAGING_ROOT / name).write_bytes(payload)

        inventory_path = STAGING_ROOT / SOURCE_INVENTORY_NAME
        inventory_path.write_bytes(inventory_bytes)
        manifest_payload = {item["path"]: item["sha256"] for item in inventory}
        manifest_payload[SOURCE_INVENTORY_NAME] = sha256_bytes(inventory_bytes)
        sha_manifest_path = STAGING_ROOT / SHA256_MANIFEST_NAME
        sha_manifest_path.write_text(
            canonical_json(manifest_payload),
            encoding="utf-8",
        )
        receipt = {
            "schema_version": "2.0.0",
            "status": "P0_P2_SOURCE_MATERIALIZED_V2",
            "producer_notebook_name": NOTEBOOK_NAME,
            "output_dataset_name": OUTPUT_DATASET_NAME,
            "output_directory": OUTPUT_DIRECTORY_NAME,
            "source_repository_commit": EXPECTED_SOURCE_REPOSITORY_COMMIT,
            "source_bundle_name": SOURCE_BUNDLE_NAME,
            "source_bundle_sha256": EXPECTED_SOURCE_BUNDLE_SHA256,
            "bundle_manifest_sha256": EXPECTED_BUNDLE_MANIFEST_SHA256,
            "source_inventory_sha256": EXPECTED_SOURCE_INVENTORY_SHA256,
            "sha256_manifest_sha256": sha256_file(sha_manifest_path),
            "source_file_count": len(inventory),
            "network_access_permitted": False,
            "credentials_present": False,
            "customer_data_present": False,
            "model_loads": 0,
            "worker_starts": 0,
            "model_requests": 0,
            "benchmark_trajectory_requests": 0,
            "external_spend": 0,
            "next_gate": "execute_metadata_only_p0_p2_source_inspection_v2",
        }
        (STAGING_ROOT / MATERIALIZATION_RECEIPT_NAME).write_text(
            canonical_json(receipt),
            encoding="utf-8",
        )
        STAGING_ROOT.replace(OUTPUT_ROOT)
    except Exception:
        if STAGING_ROOT.exists():
            for child in STAGING_ROOT.iterdir():
                child.unlink(missing_ok=True)
            STAGING_ROOT.rmdir()
        raise

    print(
        canonical_json(
            {
                "status": "P0_P2_SOURCE_MATERIALIZED_V2",
                "output_directory": str(OUTPUT_ROOT),
                "output_dataset_name": OUTPUT_DATASET_NAME,
                "source_bundle_sha256": EXPECTED_SOURCE_BUNDLE_SHA256,
                "source_file_count": len(inventory),
                "next_gate": (
                    "execute_metadata_only_p0_p2_source_inspection_v2"
                ),
            }
        )
    )


main()